In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, math, csv, datetime, time, pickle, json
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
base_dir = '/content/drive/MyDrive/fire_non_fire_data'

In [ ]:
train_dir = os.path.join(base_dir, 'train')
os.path.exists(train_dir)

In [ ]:
test_dir = os.path.join(base_dir, 'test')
os.path.exists(test_dir)

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 0.0003
THRESHOLD = 0.5
SEED = 42
EARLY_STOPPING_PATIENCE_EPOCHS = 5
REDUCE_ON_PLATEAU_LR_FACTOR = 0.5
REDUCE_ON_PLATEAU_LR_PATIENCE = 3

In [ ]:
tf.keras.utils.set_random_seed(SEED)

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=40,
    zoom_range=0.30,
    width_shift_range=0.20,
    height_shift_range=0.20,
    shear_range=0.15,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True,
    fill_mode='nearest'
)

In [ ]:
test_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0
)


In [ ]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=SEED
)

In [ ]:
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

In [ ]:
y_train = train_generator.classes

class_ids = np.unique(y_train)

weights = compute_class_weight(
    class_weight='balanced',
    classes=class_ids,
    y=y_train
)

class_weights = {
    int(class_id): float(weight)
    for class_id, weight in zip(class_ids, weights)
}

print("Class distribution:")
for name, index in train_generator.class_indices.items():
    print(f"{name}: {np.sum(y_train == index)} images")

print("\nClass weights:")
print(class_weights)

In [ ]:
## creating model

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(128),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')

])

In [ ]:
loss_function = tf.keras.losses.BinaryCrossentropy()

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

In [ ]:
model.compile(optimizer=optimizer, loss=loss_function, metrics=[tf.keras.metrics.BinaryAccuracy()])

In [ ]:
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='loss',
    factor=REDUCE_ON_PLATEAU_LR_FACTOR,
    patience=REDUCE_ON_PLATEAU_LR_PATIENCE
)

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
fit_summary = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[reduce_lr, early_stopping],
    class_weight=class_weights
)

In [ ]:
x_range = np.arange(20)

In [ ]:
train_acc = fit_summary.history['binary_accuracy']
train_loss = fit_summary.history['loss']
val_acc = fit_summary.history['val_binary_accuracy']
val_loss = fit_summary.history['val_loss']
learning_rate = fit_summary.history['learning_rate']

In [ ]:
plt.figure(figsize=(20, 7))
plt.grid()
plt.plot(x_range, train_acc, color='purple')
for x_tick, each_acc in enumerate(train_acc):
    plt.scatter(x_tick, each_acc, color='red')
    plt.text(x_tick, each_acc + 0.008, f'{each_acc:.2f}', color='blue')
_ = plt.xticks(x_range)
plt.xlabel('Epochs')
_ = plt.ylabel('Train Accuracy')
_ = plt.title('Train accuracy Plot')
plt.savefig('Train_Acc.png')

In [ ]:
plt.figure(figsize=(20, 7))
plt.grid()
plt.plot(x_range, train_loss, color='orange')
for x_tick, each_acc in enumerate(train_loss):
    plt.scatter(x_tick, each_acc, color='blue')
    plt.text(x_tick - 0.3, each_acc - 0.1, f'{each_acc:.2f}', color='green')
_ = plt.xticks(x_range)
plt.xlabel('Epochs')
_ = plt.ylabel('Train Loss')
_ = plt.title('Train Loss Plot')
plt.savefig('Train_Loss.png')

In [ ]:
plt.figure(figsize=(20, 7))
plt.grid()
plt.plot(x_range, val_acc, color='purple')
for x_tick, each_acc in enumerate(val_acc):
    plt.scatter(x_tick, each_acc, color='red')
    plt.text(x_tick, each_acc + 0.008, f'{each_acc:.2f}', color='blue')
_ = plt.xticks(x_range)
plt.xlabel('Epochs')
_ = plt.ylabel('Validation Accuracy')
_ = plt.title('Validation Accuracy Plot')
plt.savefig('Val_Acc.png')

In [ ]:
plt.figure(figsize=(20, 7))
plt.grid()
plt.plot(x_range, val_loss, color='orange')
for x_tick, each_acc in enumerate(val_loss):
    plt.scatter(x_tick, each_acc, color='blue')
    plt.text(x_tick - 0.4, each_acc - 0.14, f'{each_acc:.2f}', color='green')
_ = plt.xticks(x_range)
plt.xlabel('Epochs')
_ = plt.ylabel('Validation Loss')
_ = plt.title('Validation Loss Plot')
plt.savefig('Val_Loss.png')

In [ ]:
test_y_preds = model.predict(test_generator)

In [ ]:
test_y_pred_labels = (test_y_preds.squeeze() > THRESHOLD).astype(np.uint8)

In [ ]:
y_test = test_generator.classes

In [ ]:
train_generator.class_indices

In [ ]:
class_names = list(train_generator.class_indices.keys())

In [ ]:
print(classification_report(y_true=y_test, y_pred=test_y_pred_labels, target_names=class_names))

In [ ]:
## Convert the model to tflite format

In [ ]:
model.export('./saved_model', format='tf_saved_model')

In [ ]:
tf.keras.models.save_model(model, './model.keras')

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

with open("model.tflite", "wb") as f:
    f.write(tflite_model)